# Домашнее задание: Ревью кода + классификация ошибок

**Тема:** ИИ как первый ревьюер — фильтрация замечаний, классификация по схеме «светофор», оформление PR-комментариев.

**Преподаватель:** Владимир Языков  
**Студент:** Максим Мамалыга  
**Языки:** Python, Java, Go

---
## Подготовка

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

def color_row(val):
    c_map = {
        'blocker': 'background: #fcc',
        'major': 'background: #ffc',
        'minor': 'background: #ddd',
        'Accept': 'background: #cfc',
        'Verify': 'background: #ffc',
        'Reject': 'background: #fcc'
    }
    return c_map.get(val, '')

print("Готово")
print("pandas", pd.__version__)

---

## Пример 1 — Python: Command Injection

**Ситуация:** утилита для бэкапа, аргумент `folder` приходит из веб-формы.

In [ ]:
# Исходный код
CODE_PY = '''import os

def run_backup(folder: str) -> int:
    cmd = f"tar -czf backup.tgz {folder}"
    return os.system(cmd)'''

print(CODE_PY)

### Промпт (отправлял в ChatGPT)

In [ ]:
PROMPT_PY = '''Ты сеньор Python-разработчик. Сделай код-ревью функции бэкапа.

Контекст: функция run_backup принимает путь к папке от пользователя и подставляет в shell-команду через f-string.

Что проверить:
1. Безопасность — может ли пользователь выполнить произвольную команду?
2. Корректность — что будет, если в названии папки пробелы или спецсимволы?
3. Обработка ошибок — что если tar упал?
4. Читаемость — насколько код понятен

Ограничения:
- Не предлагай переписать на другой язык
- Не предлагай внешние библиотеки без необходимости
- Крупный рефакторинг только если обоснован

Формат: для каждого замечания пиши [BLOCKER/MAJOR/MINOR], потом описание, потом Risk, потом Fix.
Спасибо!'''

print(PROMPT_PY)

### Ответ ChatGPT (скопирован из чата)

In [ ]:
AI_PY = '''Привет! Вот мой разбор:

**[BLOCKER] Command injection через непроверенный ввод**
Переменная folder подставляется в команду через f-string. Если передать "; rm -rf /" — команда выполнится.
Risk: полная компрометация системы.
Fix: использовать subprocess.run с массивом аргументов, без shell=True.

**[MAJOR] Используется устаревший os.system**
os.system() вызывает shell, что увеличивает поверхность атаки. Лучше использовать subprocess.
Risk: лишний канал для инжекций.
Fix: subprocess.run(['tar', '-czf', 'backup.tgz', folder], check=True).

**[MAJOR] Нет обработки ошибок**
os.system возвращает код возврата, но он нигде не проверяется. Функция вернёт 0 даже при ошибке.
Risk: тихие падения бэкапа, никто не узнает.
Fix: добавить check=True или явную проверку returncode.

**[MAJOR] Нет валидации входного параметра folder**
Нет проверки, что это вообще существующая директория.
Risk: ошибки выполнения, непонятные сообщения пользователю.
Fix: проверять os.path.isdir(folder) перед выполнением.

**[MINOR] Тип возврата int вместо bool**
Небольшая несогласованность API. int работает, но более читаемо было бы bool или исключение.
Risk: низкий.
Fix: можно переделать на возврат bool, но не критично.

**[MINOR] Предлагаю использовать pathlib**
pathlib удобнее для работы с путями, но в данном случае не обязательно.
Risk: нет.
Fix: можно позже, как рефакторинг.
'''

print(AI_PY)

### Классификация замечаний

In [ ]:
tbl_py = pd.DataFrame([
    {
        'Замечание': 'Command injection',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'blocker',
        'Что делаем': 'переписать на subprocess.run с list-аргументом'
    },
    {
        'Замечание': 'Устаревший os.system',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'заменить вместе с первым пунктом'
    },
    {
        'Замечание': 'Нет обработки ошибок',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'добавить check=True, обернуть в try-except'
    },
    {
        'Замечание': 'Нет валидации folder',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'добавить isdir проверку, вернуть ошибку если путь невалиден'
    },
    {
        'Замечание': 'Тип возврата int',
        'Тип': 'сомнительное',
        'Светофор': 'Verify',
        'Приоритет': 'minor',
        'Что делаем': 'проверить контракты вызывающего кода'
    },
    {
        'Замечание': 'pathlib',
        'Тип': 'сомнительное',
        'Светофор': 'Verify',
        'Приоритет': 'minor',
        'Что делаем': 'если в проекте уже есть pathlib в других местах — принять'
    },
])

tbl_py.style.map(color_row, subset=['Приоритет']).map(color_row, subset=['Светофор'])

### Комментарии к PR

In [ ]:
PR_PY = '''
**[BLOCKER] Command injection**
Переменная folder идёт в shell без обработки. Это позволяет выполнить любую команду на сервере.
Риск: полная компрометация, удаление файлов.
Исправить: subprocess.run(['tar', '-czf', 'backup.tgz', folder], check=True).

**[MAJOR] Нет валидации и обработки ошибок**
Нет проверки, что папка существует. Код возвращает 0 даже при ошибке tar.
Риск: потеря данных без уведомления.
Исправить: добавить os.path.isdir(folder) и check=True.
'''

display(Markdown(PR_PY))

### Next steps

In [ ]:
NS_PY = '''
До merge:
- Переписать os.system -> subprocess.run с массивом
- Добавить проверку существования директории
- Завернуть в try-except

В отдельную задачу:
- Решить нужен ли pathlib (если в проекте уже есть — использовать)

Тесты:
- тест с существующей папкой
- тест с несуществующей
- тест с спецсимволами в пути
'''

display(Markdown(NS_PY))

---

## Пример 2 — Java: NullPointerException

**Ситуация:** сервис отправляет письмо, репозиторий может вернуть null.

In [ ]:
CODE_JV = '''public class MailService {
    private final UserRepository userRepository;
    private final MailClient mailClient;

    public MailService(UserRepository userRepository, MailClient mailClient) {
        this.userRepository = userRepository;
        this.mailClient = mailClient;
    }

    public void sendWelcomeEmail(Long userId) {
        User user = userRepository.findById(userId);
        mailClient.send(user.getEmail(), "Welcome!");
    }
}'''

print(CODE_JV)

### Промпт

In [ ]:
PROMPT_JV = '''Ты сеньор Java-разработчик. Проверь класс MailService.

Контекст: метод sendWelcomeEmail ищет User по ID и отправляет письмо. userRepository.findById() может вернуть null.

Что проверить:
1. Null safety — что будет если user = null?
2. Обработка ошибок — что если почта не отправилась?
3. Код-стайл — инжекция, именование

Ограничения:
- Не предлагай Lombok, Reactive и другие библиотеки
- Не менять язык
- Если нужен Optional — объясни почему

Формат: [BLOCKER/MAJOR/MINOR] описание \n Risk \n Fix
Спасибо!'''

print(PROMPT_JV)

### Ответ ChatGPT

In [ ]:
AI_JV = '''Привет! Вот что нашёл:

**[BLOCKER] NullPointerException при findById**
userRepository.findById(userId) может вернуть null, и вы сразу вызываете .getEmail() без проверки.
Risk: падение сервиса, 500 ошибка клиенту.
Fix: добавь if (user == null) { throw new UserNotFoundException(); }.

**[MAJOR] Нет обработки исключений mailClient.send()**
Если SMTP упал, исключение летит наружу без логирования.
Risk: нельзя диагностировать проблемы с почтой.
Fix: оберни в try-catch, залогируй.

**[MAJOR] User.getEmail() может вернуть null**
Если в классе User нет валидации поля email.
Risk: NPE в mailClient.send(null, ...).
Fix: добавь проверку Objects.requireNonNull(email) в конструктор User.

**[MAJOR] Предлагаю использовать Optional <User>**
Это бы явно показывало, что пользователя может не быть.
Risk: низкий, но повышает читаемость.
Fix: изменить сигнатуру на Optional<User> findById(Long id).

**[MINOR] Нейминг метода**
sendWelcomeEmail можно было бы назвать более явно, но это вкусовщина.
Risk: нет.
Fix: по желанию.
'''

print(AI_JV)

### Классификация

In [ ]:
tbl_jv = pd.DataFrame([
    {
        'Замечание': 'NPE при findById',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'blocker',
        'Что делаем': 'добавить проверку user == null'
    },
    {
        'Замечание': 'Нет обработки исключений send()',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'обернуть в try-catch с логированием'
    },
    {
        'Замечание': 'User.getEmail() может быть null',
        'Тип': 'корректное',
        'Светофор': 'Verify',
        'Приоритет': 'major',
        'Что делаем': 'проверить конструктор User — null там не допускается? Если допускается — исправить'
    },
    {
        'Замечание': 'Использовать Optional<User>',
        'Тип': 'сомнительное',
        'Светофор': 'Verify',
        'Приоритет': 'major',
        'Что делаем': 'Optional меняет контракт findById — нужно обсудить с командой и проверить всех вызывающих'
    },
    {
        'Замечание': 'Нейминг sendWelcomeEmail',
        'Тип': 'сомнительное',
        'Светофор': 'Verify',
        'Приоритет': 'minor',
        'Что делаем': 'проверить код-стайл проекта, если принято называть кратко — оставить'
    },
])

tbl_jv.style.map(color_row, subset=['Приоритет']).map(color_row, subset=['Светофор'])

### Комментарии к PR

In [ ]:
PR_JV = '''
**[BLOCKER] Проверять user на null после findById**
Метод может вернуть null, а код сразу обращается к user.getEmail() — будет NPE.
Риск: падение сервиса при каждом запросе с несуществующим userId.
Исправить: добавить if (user == null) throw new UserNotFoundException().

**[MAJOR] mailClient.send() без try-catch**
Если SMTP недоступен — исключение пробрасывается наружу, пользователь видит 500.
Риск: невозможность понять, что проблема в почте, а не в коде.
Исправить: обернуть в try-catch, залогировать.

**[MAJOR] Возможен null в email пользователя**
Если User допускает null email — mailClient.send упадёт с NPE.
Риск: ложная трассировка, будут искать проблему в MailService.
Исправить: requireNonNull(email) в конструкторе User или проверка перед send.
'''

display(Markdown(PR_JV))

### Next steps

In [ ]:
NS_JV = '''
До merge:
- Проверка user == null после findById
- try-catch вокруг send()

В отдельную задачу:
- Перейти на Optional<User> в findById (если команда согласна)
- Проверить, есть ли валидация email в User

Тесты:
- user не найден → исключение
- mailClient.send упал → проверить лог
- успешный сценарий → письмо отправлено
'''

display(Markdown(NS_JV))

---

## Пример 3 — Go: SQL-инъекция

**Ситуация:** параметр status из URL подставляется в SQL-запрос через строку.

In [ ]:
CODE_GO = '''package handlers

import (
    "database/sql"
    "fmt"
    "net/http"
)

func Orders(db *sql.DB, w http.ResponseWriter, r *http.Request) error {
    status := r.URL.Query().Get("status")
    query := fmt.Sprintf("SELECT id, total FROM orders WHERE status = '%s'", status)
    _, err := db.Query(query)
    return err
}'''

print(CODE_GO)

### Промпт

In [ ]:
PROMPT_GO = '''Ты senior Go-разработчик. Проверь обработчик заказов.

Контекст: HTTP handler Orders берёт параметр status из URL и подставляет в SQL через fmt.Sprintf.

Что проверить:
1. SQL injection — главное
2. Обработка ошибок — не утекает ли инфа о БД клиенту?
3. HTTP — возвращается ли статус-код?
4. Валидация — может ли пользователь передать что угодно?

Ограничения:
- Не предлагать ORM
- Не менять язык
- Предлагать только стандартные библиотеки

Формат: [BLOCKER/MAJOR/MINOR] \n Risk \n Fix
Спасибо!'''

print(PROMPT_GO)

### Ответ ChatGPT

In [ ]:
AI_GO = '''Привет! Вот мой разбор:

**[BLOCKER] SQL injection через fmt.Sprintf**
Пользовательский ввод прямо в строку SQL. Можно передать status=1' OR '1'='1.
Risk: утечка всех заказов.
Fix: используй параметризованный запрос: db.Query("SELECT ... WHERE status = ?", status).

**[MAJOR] Утечка информации через return err**
Клиент получает сырую ошибку из БД. Это раскрывает структуру таблиц.
Risk: помощь злоумышленнику.
Fix: логируй, верни generic 500.

**[MAJOR] Нет HTTP статус-кода**
Нигде не установлен 200 OK при успехе.
Risk: клиенты не знают, что запрос выполнен.
Fix: добавь w.WriteHeader(200) или эквивалент.

**[MAJOR] Нет валидации status**
Принимается любое значение без проверки.
Risk: лишние запросы к БД, потенциально некорректные данные.
Fix: проверяй по белому списку ["pending", "shipped", "delivered"], иначе 400.

**[MINOR] Нормализация регистра status**
Если пользователь передаст "PENDING" вместо "pending" — запрос ничего не вернёт.
Risk: низкий, но может сбить с толку.
Fix: strings.ToLower(status) перед валидацией.
'''

print(AI_GO)

### Классификация

In [ ]:
tbl_go = pd.DataFrame([
    {
        'Замечание': 'SQL injection',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'blocker',
        'Что делаем': 'переписать на parameterized query'
    },
    {
        'Замечание': 'Утечка ошибки БД клиенту',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'заменить return err на лог + 500'
    },
    {
        'Замечание': 'Нет HTTP статуса при успехе',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'добавить WriteHeader(200)'
    },
    {
        'Замечание': 'Нет валидации status',
        'Тип': 'корректное',
        'Светофор': 'Accept',
        'Приоритет': 'major',
        'Что делаем': 'добавить whitelist + 400'
    },
    {
        'Замечание': 'Нормализация регистра status',
        'Тип': 'корректное',
        'Светофор': 'Verify',
        'Приоритет': 'minor',
        'Что делаем': 'проверить ожидаются ли значения в другом регистре'
    },
])

tbl_go.style.map(color_row, subset=['Приоритет']).map(color_row, subset=['Светофор'])

### Комментарии к PR

In [ ]:
PR_GO = '''
**[BLOCKER] SQL injection в status**
Параметр status подставляется в SQL через fmt.Sprintf без экранирования.
Риск: утечка всех заказов через 1' OR '1'='1.
Исправить: parameterized query.

**[MAJOR] Не возвращать err клиенту**
Если db.Query вернул ошибку — она уходит в HTTP-ответ. Клиент видит структуру БД.
Риск: information disclosure.
Исправить: залогировать, вернуть generic 500.

**[MAJOR] Нет WriteHeader(200)**
При успехе не отправляется статус-код.
Риск: клиент ждёт OK, а получает пустой ответ.
Исправить: добавить w.WriteHeader(200).

**[MAJOR] Валидация status**
Любое значение идёт в запрос.
Риск: лишняя нагрузка на БД.
Исправить: whitelist + 400 Bad Request.
'''

display(Markdown(PR_GO))

### Next steps

In [ ]:
NS_GO = '''
До merge:
- Parameterized query вместо fmt.Sprintf
- Валидация status по whitelist
- Логирование ошибок БД, return generic 500
- WriteHeader(200)

В отдельную задачу:
- Решить нужен ли ToLower для status

Тесты:
- валидный статус → 200
- невалидный статус → 400
- попытка injection → безопасный запрос
- ошибка БД → 500 + лог
'''

display(Markdown(NS_GO))

---

## Сводная таблица

In [ ]:
all_tbl = pd.concat([
    tbl_py.assign(Пример='1. Python'),
    tbl_jv.assign(Пример='2. Java'),
    tbl_go.assign(Пример='3. Go'),
])

summary = all_tbl.groupby(['Пример', 'Светофор']).size().unstack(fill_value=0)
display(Markdown("### Распределение по светофору"))
display(summary)


display(Markdown("### Проверка выполнения требований"))

check = pd.DataFrame([
    ['[x]', '3 примера (Python, Java, Go)'],
    ['[x]', 'У каждого: код, промпт, ответ ИИ, классификация, PR, next'],
    ['[x]', 'Промпты: контекст, критерии, ограничения, формат'],
    ['[x]', 'Классификация: тип, светофор, приоритет, что делаем'],
    ['[x]', 'PR: что не так -> риск -> как исправить -> приоритет'],
    ['[x]', 'Next steps: до merge, отдельная задача, тесты'],
], columns=['', 'Критерий'])
display(check)